In [7]:
# 安装 Nuvolos 官方点名支持的原生 Whisper 库
!pip install openai-whisper pandas tqdm

print("\n[SUCCESS] 官方原生 ASR 运行环境铺设完毕！")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.0 MB/s  0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=006040238f5c13bcb9849451d3819c335771b0a34c72a003fbcce8ab00202bbf
  Stored in directory: /tmp/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai-whisper]m [openai-whisper]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[SUCCESS] 官方原生 ASR 运行环境铺设完毕！


In [ ]:
# ======================================================================
# 航母级总攻单元格：一键自动化解压 ＋ 纯物理路径对齐 ＋ 原生流长跑
# ======================================================================
import os
import sys
import pandas as pd
import torch
import gc

# ------------------ STEP 1: 现场闪击解压 ------------------
print("="*70)
print("[步骤 1] 粮草先行：正在全力当场解压最新上传的 data1.1.tar...")
print("="*70, flush=True)

# 🚨 如果你的压缩包在当前目录下，直接运行这行。如果它在 data1/ 目录下，请改成 !tar -xf data1/data1.1.tar
!tar -xf data1.1.tar

print("\n[SUCCESS ✅] 音频数据集当场释放完毕！")

In [1]:
# ======================================================================
# 🚀 导师终极重构：针对全新解压 data1.1/ 目录硬对齐高容灾流水线
# ======================================================================
import pandas as pd
import os
import torch
import whisper
from tqdm.notebook import tqdm

def run_nuvolos_final_pure_path_pipeline(metadata_path, output_path):
    print("\n" + "="*70)
    print(f"[Nuvolos 终极会战] 针对 data1.1/ 现场硬对齐高容灾流水线启动...")
    print("= 核心修复：彻底抛弃 data1/ 外壳，硬核钉死在 data1.1/{dialect}/{filename}")
    print("="*70, flush=True)
    
    if not os.path.exists(metadata_path):
        print(f"[CRASH] 找不到输入的元数据表: {metadata_path}")
        return
        
    df_master = pd.read_csv(metadata_path)
    
    # 1. 强力洗表：把刚才因为路径 Bug 导致写坏的假 FILE_NOT_FOUND 全量擦除，重新排队补跑
    if os.path.exists(output_path):
        try:
            df_existing = pd.read_csv(output_path)
            if 'pred_text_whisper_large' in df_existing.columns:
                # 💡 只保留真正跑出文字的行，把刚才写坏的错误占位符全部抹除
                df_clean = df_existing[
                    df_existing['pred_text_whisper_large'].notna() & 
                    (~df_existing['pred_text_whisper_large'].astype(str).str.contains("FILE_NOT_FOUND|ERROR"))
                ].copy()
                
                df_clean.to_csv(output_path, index=False, encoding='utf-8')
                completed_paths = set(df_clean['audio_path'].dropna())
                print(f"[自愈装置 🟢] 成功从硬盘 CSV 中清空了刚才因路径错位写坏的假死行。")
                print(f"[断点扫描] 当前已落实的真实黄金成果数: {len(completed_paths)} 条。")
        except Exception as e:
            print(f"[WARN] 盘点历史失败: {e}")
            completed_paths = set()
    else:
        completed_paths = set()

    df_todo = df_master[~df_master['audio_path'].isin(completed_paths)].copy()
    total_todo = len(df_todo)
    
    if total_todo == 0:
        print("\n[🎉 完美通关] 全量数据已全部高质量落盘！无须重复运行！")
        return
        
    print(f"[任务大盘点] 总任务: {len(df_master)} 条 | 本次实际要抢救/补跑的任务: {total_todo} 条")
    print("-" * 70, flush=True)

    # 2. 【物理硬核路径组装：精准扣杀】
    # 既然你证实了现在直接就是 data1.1/zh/ 开头
    # 我们直接强行提取最右边的文件名和方言文件夹，硬拼到 data1.1/ 目录下！
    def build_perfect_path(row):
        filename = os.path.basename(str(row['audio_path'])) # 提取出 "ch_zh_0000.wav"
        dialect = str(row['dialect']).lower().strip()       # 提取出 "zh"
        return f"data1.1/{dialect}/{filename}".replace("\\", "/")

    df_todo['real_audio_path'] = df_todo.apply(build_perfect_path, axis=1)

    # 3. 哨兵首行绝对通路自检
    first_p = df_todo['real_audio_path'].iloc[0]
    print(f"[通路抽查] 针对断点后第一条任务，代码当前重组出来的物理路径为:\n   👉 '{first_p}'")
    
    if not os.path.exists(first_p):
        print(f"\n[🚨 熔断报错] 依然找不到文件！请在下面新开单元格运行 !ls -l {first_p} 确认此音频位置。")
        return
    print("[验证通过 🟢] 物理血脉百分之百彻底打通！安全锁全面解除。")
    print("-" * 70, flush=True)

    # 4. 显存清场并载入官方原生模型
    import gc
    torch.cuda.empty_cache()
    gc.collect()

    try:
        model = whisper.load_model("large-v3", device="cuda")
        print("[SUCCESS ✨] OpenAI 原生 Whisper-Large-v3 模型及底座成功装载！")
    except Exception as e:
        print(f"[🚨 OOM 阻截提示] 显存卡死，请点击 Kernel -> Restart Kernel 刷新显卡！详情: {e}")
        return

    print(f"\n[INFO] 正式开启长跑，结果单条追加冷落地...", flush=True)
    
    file_exists_now = os.path.exists(output_path) and os.path.getsize(output_path) > 0
    write_header = not file_exists_now
    counter = 0
    
    for idx, row in tqdm(df_todo.iterrows(), total=total_todo, desc="Nuvolos 真正全速长跑"):
        audio_file = row['real_audio_path']
        
        if not os.path.exists(audio_file):
            pred_text = "FILE_NOT_FOUND"
        else:
            try:
                result = model.transcribe(audio_file)
                pred_text = result["text"].strip()
            except Exception as e:
                pred_text = f"ERROR_{str(e)[:20]}"
        
        # 写入大表时，依然保持原始 master_metadata 格式，不破坏你的相对路径数据
        single_record = {
            "sentence_id": row['sentence_id'],
            "dialect": row['dialect'],
            "audio_path": row['audio_path'],  
            "ref_high_german": row['ref_high_german'],
            "ref_swiss_german": row['ref_swiss_german'],
            "pred_text_whisper_large": pred_text
        }
        
        df_single = pd.DataFrame([single_record])
        df_single.to_csv(output_path, mode='a', index=False, header=write_header, encoding='utf-8')
        write_header = False  
        
        counter += 1
        if counter % 10 == 0:
            print(f"   [实时进度看板] 已成功转写并安全落盘: {counter} / {total_todo} 条...", flush=True)

    print("\n" + "="*70)
    print("[大获全胜 🎉] 历经千辛万苦，全量高容灾流水线终于圆满大收官！")
    print(f" 干净完整的成果大表保存在 -> {output_path}")
    print("="*70 + "\n")


if __name__ == "__main__":
    INPUT_CSV = "master_metadata.csv"
    OUTPUT_CSV = "master_metadata_whisper_large_completed.csv"
    
    run_nuvolos_final_pure_path_pipeline(metadata_path=INPUT_CSV, output_path=OUTPUT_CSV)


[Nuvolos 终极会战] 针对 data1.1/ 现场硬对齐高容灾流水线启动...
= 核心修复：彻底抛弃 data1/ 外壳，硬核钉死在 data1.1/{dialect}/{filename}
[自愈装置 🟢] 成功从硬盘 CSV 中清空了刚才因路径错位写坏的假死行。
[断点扫描] 当前已落实的真实黄金成果数: 166 条。
[任务大盘点] 总任务: 30921 条 | 本次实际要抢救/补跑的任务: 30755 条
----------------------------------------------------------------------
[通路抽查] 针对断点后第一条任务，代码当前重组出来的物理路径为:
   👉 'data1.1/be/ch_be_0021.wav'
[验证通过 🟢] 物理血脉百分之百彻底打通！安全锁全面解除。
----------------------------------------------------------------------
[SUCCESS ✨] OpenAI 原生 Whisper-Large-v3 模型及底座成功装载！

[INFO] 正式开启长跑，结果单条追加冷落地...


Nuvolos 真正全速长跑:   0%|          | 0/30755 [00:00<?, ?it/s]

   [实时进度看板] 已成功转写并安全落盘: 10 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 20 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 30 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 40 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 50 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 60 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 70 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 80 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 90 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 100 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 110 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 120 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 130 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 140 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 150 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 160 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 170 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 180 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 190 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 200 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 210 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 220 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 230 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 240 / 30755 条...
   [实时进度看板] 已成功转写并安全落盘: 2